In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import signal, fft, stats
from scipy.ndimage import gaussian_filter
from scipy.stats import gaussian_kde
import os
import warnings
import cfad
from bg_atlasapi import BrainGlobeAtlas
import json
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from joblib import Parallel, delayed


# --- 1. GLOBAL CONFIGURATION ---
PARAMS = {
    # Hardware Specs
    'fs_raw': 30000,          # Acquisition Rate (Hz)
    'fs_lfp': 1000,           # Downsampled Rate (Hz)
    'voltage_scaling': 0.195, # uV per bit
    'probe_tip_offset': 720,  # Microns from tip with no channels

    # Analysis Windows
    'window_lfp': (-0.1, 0.5),    # Time window around trigger (s)
    'window_visual': (0.075, 0.150),# Window to find sSC sink (75 - 150ms post-stim)
    'bin_size_um': 15,            # Spatial bin for virtual probe
    
    # Bad Channel Thresholds
    'bad_ch_thresh': {
        'rms_low': 1.0,        # Dead (<1uV)
        'rms_high_factor': 15, # Noisy (>15x median)
        'corr_min': 0.6        # Artifact (<0.6 correlation to brain)
    },
    
    # Spike Statistics
    'spike_win_pre': (-0.2, 0.0), # Baseline window (s)
    'spike_win_post': (0, 0.2)# Visual response window (s)
}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'axes.linewidth': 1.2,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,  # Ensures fonts are editable in Adobe Illustrator
    'ps.fonttype': 42,
    'axes.spines.top': False,   # "Despining" for a cleaner look
    'axes.spines.right': False
})
Group = [None, "Vehicle", "Vehicle", "Psilocybin", "Vehicle", "Psilocybin", "Vehicle", "Psilocybin", "Psilocybin", "Vehicle", "Psilocybin", "Vehicle"]

# --- 2. SIGNAL PROCESSOR CLASS ---
class NeuropixelsProcessor:
    """
    processor class, processes chunks at a time to reduce computation time from doing it to the whole recording
    init defines self, n channels, parameters for recording, shifts for deskewing and the parametors for butterworth filter
    """
    def __init__(self, n_channels, params):
        self.n_ch = n_channels
        self.p = params
        #defines numerator (b) and denominator (a) for butterworth polynomial
        self.b, self.a = signal.butter(3, 300, btype='low', fs=params['fs_raw'])
        self.bad_mask = None
        #defines shifts for deskewing
        self.shifts = np.arange(n_channels) / n_channels

    def detect_bad_channels(self, raw_chunk):
        """
        detects bad channels of chunk
            1. scales to voltage from int16
            2. computes rms
            3. computes correlations against common median
            4. defines bad channels via threshold in params
            5. returns bad channels/mask as self.bad_mask, and counts of bad channels
        """
        #scale to volts
        data = raw_chunk.astype(np.float32) * self.p['voltage_scaling']
        #computer rms
        rms = np.sqrt(np.mean(data**2, axis=1))
        median_rms = np.nanmedian(rms)
        #correlate to common median
        global_sig = np.median(data, axis=0)
        corrs = np.zeros(self.n_ch)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for i in range(self.n_ch):
                if rms[i] > 0.1: corrs[i] = np.corrcoef(data[i], global_sig)[0,1]
        #define threshold
        t = self.p['bad_ch_thresh']
        #assing bad channel mask
        self.bad_mask = (rms < t['rms_low']) | (rms > median_rms * t['rms_high_factor']) | (corrs < t['corr_min'])
        return {'bad_mask': self.bad_mask, 'counts': (np.sum(rms < t['rms_low']), np.sum(rms > median_rms * t['rms_high_factor']), np.sum(corrs < t['corr_min']))}
 
    def process_chunk(self, raw_chunk, clean=True):
        """
        chunk processor
            1. scales to volatage
            2. clean = false then applies butterworth filter and downsamples, return causes function break i.e stops doing cleaning if already clean
            3. clean = True cleans data by:
                1. deskewing data
                2. setting bad channels to nan
                3. detrending and cmr
                4. applies butterworth filter
                5. returns downsampled data
        thus end return of chunk is same, resulting in a processed lfp for the chunk, but this method handles if the chunk has already been cleaned
        """
        data = raw_chunk.astype(np.float32) * self.p['voltage_scaling']
        if not clean:
            data = signal.filtfilt(self.b, self.a, data, axis=1)
            q = int(self.p['fs_raw'] / self.p['fs_lfp'])
            return data[:, ::q]

        # Deskew
        n_t = data.shape[1]
        f = fft.rfft(data, axis=1)
        freqs = fft.rfftfreq(n_t)
        shift = np.exp(-1j * 2 * np.pi * freqs[None, :] * self.shifts[:, None])
        data = fft.irfft(f * shift, n=n_t, axis=1)
        
        # Mask Bad
        if self.bad_mask is not None: data[self.bad_mask, :] = np.nan
            
        # Detrend & CMR
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            data = data - np.nanmedian(data, axis=1, keepdims=True)
            data = data - np.nanmedian(data, axis=0, keepdims=True)
            
        # Filter
        data = np.nan_to_num(data, nan=0.0) 
        data = signal.filtfilt(self.b, self.a, data, axis=1)
        if self.bad_mask is not None: data[self.bad_mask, :] = np.nan
            
        q = int(self.p['fs_raw'] / self.p['fs_lfp'])
        return data[:, ::q]


# --- 3. VIRTUAL PROBE HELPER ---

def create_virtual_probe(data, channel_depths):
    """
    Creates a virtual 1D probe by averaging electrodes that share exact 
    vertical depth coordinates, natively ignoring NaN-masked bad channels.
    
    Parameters:
    data (np.ndarray): 2D array of shape (n_channels, n_samples) containing LFP data.
    channel_depths (np.ndarray): 1D array of shape (n_channels,) with integer depth values.
    
    Returns:
    virtual_data (np.ndarray): 1D collapsed probe data (n_unique_depths, n_samples).
    unique_depths (np.ndarray): The specific integer depth coordinates.
    """
    # 1. Enforce integer matching for precise spatial filtering
    depths_int = np.asarray(channel_depths, dtype=int)
    unique_depths = np.sort(np.unique(depths_int))
    n_depths = len(unique_depths)
    
    # 2. Initialize output
    virtual_data = np.zeros((n_depths, data.shape[1]))
    
    # 3. Collapse horizontally, letting np.nanmean handle missing data natively
    for i, depth in enumerate(unique_depths):
        mask = (depths_int == depth)
        depth_slice = data[mask, :]
        
        # Only average if the slice exists and isn't 100% NaN (all coplanar channels dead)
        if np.any(mask) and not np.all(np.isnan(depth_slice)):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                virtual_data[i, :] = np.nanmean(depth_slice, axis=0)
        else:
            virtual_data[i, :] = np.nan
            
    # 4. Patch any depths where all channels were marked as bad
    df_virt = pd.DataFrame(virtual_data)
    virtual_data = df_virt.interpolate(method='linear', axis=0, limit_direction='both').values
    
    return virtual_data, unique_depths

def get_depth_map(ks_dir):
    """
    uses ks_dir to return depth_map of channels indexed by id such as depth_map[cid] = depth
    used to sort channels by depth
    returns depthmap
    """
    chan_map = np.load(os.path.join(ks_dir, 'channel_map.npy')).flatten()
    chan_pos = np.load(os.path.join(ks_dir, 'channel_positions.npy'))
    depth_map = np.full(384, np.nan)
    for i, cid in enumerate(chan_map):
        if cid < 384: 
            depth_map[cid] = chan_pos[i, 1]
    return depth_map
    
def get_depth_map_full(ks_dir):
    """
    uses ks_dir to return depth_map of channels indexed by id such as depth_map[cid] = depth
    used to sort channels by depth
    returns depthmap
    """
    chan_map = np.load(os.path.join(ks_dir, 'channel_map.npy')).flatten()
    chan_pos = np.load(os.path.join(ks_dir, 'channel_positions.npy'))
    depth_map = np.zeros((384, 2))
    for i, cid in enumerate(chan_map):
        if cid < 384: 
            depth_map[cid] = chan_pos[i]
    return depth_map

def avg_lfp_by_trigger(data, triggers, params, processor):
    """
    averages out lfp by trigger using params
    designed to use memmap as input for data, in theory can be used with any binary/numpy data of shape [samples, channels]
    uses flattened triggers ie. [trigger 1, trigger 2] etc, can not yet handle trials, will implement in future (or write seperate func)
    in theory could define epochs outside of function and interate over trials to produce
    returns mean_cleaned lfp data over trigger window
    needs specific processor for this recording as part of input, in theory can be logiced out I recon, however as redundency (different recordings may use different processors)
    """
    
    def _process_single_trigger(trig):
            s = int(trig + params['window_lfp'][0] * params['fs_raw'])
            e = int(trig + params['window_lfp'][1] * params['fs_raw'])
            
            # Return None if out of bounds so we can filter it later
            if s < 0 or e > data.shape[0]: 
                return None
                
            raw_trace = data[s:e, :384].T
            return processor.process_chunk(raw_trace, clean=True)
    
    # 2. Execute the loop in parallel across all CPU cores
    epochs_clean_raw = Parallel(n_jobs=-1, prefer="threads")(
        delayed(_process_single_trigger)(trig) for trig in triggers
    )
    
    # 3. Filter out the skipped (out of bounds) triggers
    epochs_clean = [epoch for epoch in epochs_clean_raw if epoch is not None]
            
    # 4. Original mathematical summary remains identical
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mean_clean = np.nanmean(epochs_clean, axis=0)
        
        sem_clean = stats.sem(epochs_clean, axis=0, nan_policy='omit')
        
        if np.ma.isMaskedArray(sem_clean):
            sem_clean = sem_clean.filled(np.nan)
            
    return mean_clean, sem_clean
def save_pub_fig(fig, suffix, mouse, save_dir):
    """
    Saves a figure in both high-res PNG and vector PDF formats.
    """
    save_dir = save_dir + r"\plots"
    if not save_dir:
        return
        
    os.makedirs(save_dir, exist_ok=True)
    base_name = f"Mouse{mouse}"
    fname = f"{base_name}_{suffix}"
    
    png_path = os.path.join(save_dir, f"{fname}.png")
    pdf_path = os.path.join(save_dir, f"{fname}.pdf")
    
    # bbox_inches='tight' prevents axis labels from being cut off
    fig.savefig(png_path, dpi=300, bbox_inches='tight', transparent=False)
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight', transparent=True)
    print(f"Saved: {fname}")
    plt.close(fig) # Frees memory
    
def calculate_spiking_mua_depth(spike_times, spike_positions, triggers, fs=30000, 
                                pre_stim=200, post_stim=800, time_bin_ms=5, depth_bin_um=20):
    """
    Calculates trial-averaged, depth-resolved MUA from discrete spike times.
    
    Parameters:
    spike_times (np.array): Array of spike timestamps in samples.
    spike_positions (np.array): Nx2 array where col 1 is the y-coordinate (depth).
    triggers (np.array): Array of stimulus onset timestamps in samples.
    fs (int): Sampling rate in Hz.
    pre_stim (float): Milliseconds to extract before stimulus.
    post_stim (float): Milliseconds to extract after stimulus.
    time_bin_ms (float): Resolution of the time axis in ms.
    depth_bin_um (float): Resolution of the depth axis in um.
    
    Returns:
    mua_map (np.ndarray): 2D smoothed array (depth x time) of firing rates.
    time_ax (np.ndarray): 1D array of time bin centers.
    depth_ax (np.ndarray): 1D array of depth bin centers.
    """
    print("Calculating depth-resolved Spike Density MUA...")
    
    # Extract just the depths
    spike_depths = spike_positions[:, 1]
    
    # Convert window parameters from ms to samples
    pre_samples = int((pre_stim / 1000.0) * fs)
    post_samples = int((post_stim / 1000.0) * fs)
    
    # Define physical bins for the 2D histogram
    # We use min/max of the actual spike positions to cover the whole probe
    min_depth = np.floor(np.min(spike_depths) / depth_bin_um) * depth_bin_um
    max_depth = np.ceil(np.max(spike_depths) / depth_bin_um) * depth_bin_um
    
    depth_edges = np.arange(min_depth, max_depth + depth_bin_um, depth_bin_um)
    time_edges = np.arange(-pre_stim, post_stim + time_bin_ms, time_bin_ms)
    
    # Initialize an empty 2D histogram (Depth x Time)
    total_hist = np.zeros((len(depth_edges)-1, len(time_edges)-1))
    
    # Iterate through each checkerboard reversal trial
    for trigger in triggers:
        window_start = trigger - pre_samples
        window_end = trigger + post_samples
        
        # Find spikes that occurred within this specific trial window
        mask = (spike_times >= window_start) & (spike_times <= window_end)
        trial_spikes = spike_times[mask]
        trial_depths = spike_depths[mask]
        
        # Convert spike times to relative milliseconds from stimulus onset
        relative_times_ms = ((trial_spikes - trigger) / fs) * 1000.0
        
        # Bin the spikes for this trial in 2D space (time and depth)
        trial_hist, _, _ = np.histogram2d(
            trial_depths, relative_times_ms, 
            bins=[depth_edges, time_edges]
        )
        total_hist += trial_hist
        
    # Convert absolute spike counts to Average Firing Rate (Spikes/Second/Trial)
    num_trials = len(triggers)
    bin_width_seconds = time_bin_ms / 1000.0
    mua_rate = total_hist / (num_trials * bin_width_seconds)
    
    # Apply a 2D Gaussian filter to smooth the discrete bins into a continuous density map
    # Sigma is given in units of bins (e.g., sigma=1 smooths over adjacent bins)
    mua_smoothed = gaussian_filter(mua_rate, sigma=(1.5, 2.0))
    
    # Calculate bin centers for accurate plotting axes
    depth_ax = depth_edges[:-1] + (depth_bin_um / 2)
    time_ax = time_edges[:-1] + (time_bin_ms / 2)
    
    print("MUA Spiking map generated.")
    return mua_smoothed, time_ax, depth_ax




def save_scaled_lfp_to_dat(lfp_array, file_path, sampling_rate):
    """
    Saves a voltage-scaled LFP numpy array to a flat binary .dat file,
    safely creating parent directories if they do not exist, and creates
    a companion JSON file.
    """
    
    # 1. Verify data type
    if lfp_array.dtype != np.float32:
        lfp_array = lfp_array.astype(np.float32)
        
    # 2. Extract properties to conserve
    metadata = {
        "dtype": str(lfp_array.dtype),
        "shape": lfp_array.shape,
        "sampling_rate_hz": sampling_rate,
        "is_voltage_scaled": True
    }
    
    # 3. Ensure the parent directory exists (THE FIX)
    parent_directory = os.path.dirname(file_path)
    if parent_directory:
        os.makedirs(parent_directory, exist_ok=True)
    
    # 4. Define file paths (adding .dat extension if missing for clarity)
    if not file_path.endswith('.dat'):
        file_path = f"{file_path}.dat"
        
    base_name = os.path.splitext(file_path)[0]
    json_path = f"{base_name}.json"
    
    # 5. Write the binary data directly to disk
    lfp_array.tofile(file_path)
    
    # 6. Write the metadata companion file
    with open(json_path, 'w') as f:
        json.dump(metadata, f, indent=4)
        
    print(f"Saved LFP binary to {file_path}")
    print(f"Saved LFP metadata to {json_path}")
    
def load_scaled_lfp_from_dat(file_path):
    # 1. Load metadata to get shape and dtype
    base_name = os.path.splitext(file_path)[0]
    with open(f"{base_name}.json", 'r') as f:
        metadata = json.load(f)
        
    shape = tuple(metadata["shape"])
    dtype = np.dtype(metadata["dtype"])
    
    # 2. Memory-map the flat binary file back into a NumPy array
    lfp_array = np.memmap(file_path, dtype=dtype, mode='r', shape=shape)
    
    return lfp_array, metadata["sampling_rate_hz"]

In [2]:
def run_sc_pipeline(file_path, ks_dir, triggers, mouse, save_dir=None, force_compute = False):
    print(f"--- PROCESSING: {os.path.basename(file_path)} ---")
    
    # 1. LOAD GEOMETRY
    print("Step 1: Loading Probe Geometry...")
    depth_map = get_depth_map(ks_dir)
    
    
    # 2. CALIBRATION
    print("Step 2: Calibrating (Detecting Bad Channels)...")
    processor = NeuropixelsProcessor(384, PARAMS)
    mm = np.memmap(file_path, dtype='int16', mode='r', 
                   shape=(os.path.getsize(file_path)//(385*2), 385))
    
    calib_start = 30000 * 60 
    calib_chunk = mm[calib_start:calib_start+60000, :384].T
    calib_stats = processor.detect_bad_channels(calib_chunk)
    print(f"  > Flagged {np.sum(calib_stats['bad_mask'])} Bad Channels")

    
    # 3. SAMPLE VISUALIZATION (Defining sample_raw!)
    print("Step 3: Extracting Diagnostic Sample...")
    # Grab 1 second of data near the first trigger
    samp_start = int(triggers[0])
    samp_end = samp_start + 30000 
    samp_chunk = mm[samp_start:samp_end, :384].T
    
    # THIS IS WHERE sample_raw IS DEFINED
    sample_raw = processor.process_chunk(samp_chunk, clean=False)
    sample_clean = processor.process_chunk(samp_chunk, clean=True)
    
    sample_virtual, depth_axis_sample = create_virtual_probe(sample_clean, depth_map)

    
    # 4. TRIAL AVERAGING
    mean_path =save_dir + r"\data\meanLFP\meanLFP.dat"
    if os.path.isfile(mean_path) and not force_compute:
        print("Mean already computed, loading in as memmap")
        mean_clean, mean_meta = load_scaled_lfp_from_dat(mean_path)
        mean_sem = np.load(save_dir + r"\data\meanLFP\meanLFPSEM.np.npy")
    else:
        print(f"Step 4: Averaging {len(triggers)} Trials...")
        mean_clean, mean_sem = avg_lfp_by_trigger(mm, triggers, PARAMS, processor)
        mean_dir = save_dir + r"\data\meanLFP\meanLFP"
        save_scaled_lfp_to_dat(mean_clean, mean_dir, PARAMS['fs_lfp'])    
        np.save(save_dir + r"\data\meanLFP\meanLFPSEM.np", mean_sem)
        
    # 5. VIRTUAL PROBE (AVERAGE)
    print("Step 5: Creating Virtual Probe (Average)...")
    virt_lfp_final, depth_axis = create_virtual_probe(mean_clean, depth_map )
    mean_virtual_dir = save_dir + r"\data\mean_virtualLFP\mean_virtualLFP"
    save_scaled_lfp_to_dat(virt_lfp_final, mean_virtual_dir, PARAMS['fs_lfp'])

    
    # 6. ANATOMY & CSD
    print("Step 6: Calculating CSD...")
    time_ax = np.linspace(*PARAMS['window_lfp'], mean_clean.shape[1])
    t_mask = (time_ax >= PARAMS['window_visual'][0]) & (time_ax <= PARAMS['window_visual'][1])
    min_profile = np.min(virt_lfp_final[:, t_mask], axis=1)
    depth_mask = depth_axis > 2000 
    valid_depths = depth_axis[depth_mask]
    valid_profile = min_profile[depth_mask]
    ssc_sink_depth = valid_depths[np.argmin(valid_profile)]
    
    lfp_smooth = gaussian_filter(virt_lfp_final, sigma=[5, 0])
    csd = -1 * np.diff(lfp_smooth, n=2, axis=0)
    depth_csd = depth_axis[1:-1]
    print(f"  > sSC Center: {ssc_sink_depth:.1f} um")

    #csd_dir = save_dir + r"\csd"
    #save_scaled_lfp_to_dat(csd, csd_dir)
    

    #Calculate psd
    signal_length = virt_lfp_final.shape[1]  # Assuming axis 1 is time; should equal 600
    
    # Dividing by 2 means a 300-sample window, allowing for roughly 3 overlapping segments
    nperseg_adjusted = signal_length // 2 
    
    # Calculate PSD with the adjusted parameters
    fs = PARAMS['fs_lfp']
    frequencies, psd = signal.welch(
        virt_lfp_final, 
        fs=fs, 
        window='hann', 
        nperseg=nperseg_adjusted,      
        noverlap=nperseg_adjusted // 2,  
        axis=1           
    )
    
    # 7. SPIKES
    print("Step 7: Integrating Spikes...")
    st = np.load(os.path.join(ks_dir, 'spike_times.npy')).flatten()
    sc = np.load(os.path.join(ks_dir, 'spike_clusters.npy')).flatten()
    sp = np.load(os.path.join(ks_dir, 'spike_positions.npy'))
    
    df_units = pd.DataFrame({'cid': sc, 'depth': sp[:, 1]})
    unit_depths = df_units.groupby('cid')['depth'].mean()
    
    unit_stats = []
    for cid in unit_depths.index:
        u_spikes = st[sc==cid]
        #if len(u_spikes) < 20: continue 
        
        counts_pre, counts_post = [], []
        for trig in triggers:
            n_pre = np.searchsorted(u_spikes, trig + PARAMS['spike_win_pre'][1]*PARAMS['fs_raw']) - \
                    np.searchsorted(u_spikes, trig + PARAMS['spike_win_pre'][0]*PARAMS['fs_raw'])
            n_post = np.searchsorted(u_spikes, trig + PARAMS['spike_win_post'][1]*PARAMS['fs_raw']) - \
                     np.searchsorted(u_spikes, trig + PARAMS['spike_win_post'][0]*PARAMS['fs_raw'])
            counts_pre.append(n_pre)
            counts_post.append(n_post)
        
        counts_pre, counts_post = np.array(counts_pre), np.array(counts_post)
        
        if np.all(counts_pre == counts_post) or (np.sum(counts_pre)+np.sum(counts_post)==0):
            p_val = 1.0
        else:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    _, p_val = stats.wilcoxon(counts_pre, counts_post)
            except ValueError: p_val = 1.0
        
        m_pre = np.mean(counts_pre) / 0.2
        m_post = np.mean(counts_post) / 0.2
        mod = (m_post - m_pre)/(m_post + m_pre + 1e-9)
        
        unit_stats.append({'cluster_id': cid, 'depth': unit_depths[cid], 'p_value': p_val, 'modulation': mod, 'sig': p_val<0.05})
        
    df_res = pd.DataFrame(unit_stats)
    
    
    
    # --- VISUALIZATION & SAVING LOGIC ---
    print("Generating Standalone Plots...")
    
    # ==========================================
    # Plot 1: Continuous Raw LFP
    # ==========================================
    fig1, ax0 = plt.subplots(figsize=(8, 6))
    sorted_raw = sample_raw[np.argsort(depth_map), :] 
    vm_raw = np.percentile(np.abs(sorted_raw), 99)
    
    im0 = ax0.imshow(sorted_raw, aspect='auto', cmap='RdBu_r', origin='lower', 
                     vmin=-vm_raw, vmax=vm_raw,
                     extent=[0, 1000, np.min(depth_map), np.max(depth_map)])
    ax0.set_title("Continuous Raw Data (1 sec)")
    ax0.set_xlabel("Time (ms)")
    ax0.set_ylabel("Depth (µm)")
    fig1.colorbar(im0, ax=ax0, label="Voltage (mV)")
    save_pub_fig(fig1, "1_Raw_Continuous", mouse, save_dir)
    
    # ==========================================
    # Plot 2: Continuous Virtual Probe
    # ==========================================
    fig2, ax1 = plt.subplots(figsize=(8, 6))
    vm_virt = np.percentile(np.abs(sample_virtual), 99)
    
    im1 = ax1.imshow(sample_virtual, aspect='auto', cmap='RdBu_r', origin='lower', 
                     vmin=-vm_virt, vmax=vm_virt,
                     extent=[0, 1000, depth_axis_sample[0], depth_axis_sample[-1]])
    ax1.set_title("Continuous Virtual Probe Data")
    ax1.set_xlabel("Time (ms)")
    ax1.set_ylabel("Depth (µm)")
    fig2.colorbar(im1, ax=ax1, label="Voltage (mV)")
    save_pub_fig(fig2, "2_Virtual_Continuous", mouse, save_dir)
    
    # ==========================================
    # Plot 3: Averaged Virtual LFP (Evoked)
    # ==========================================
    fig3, ax2 = plt.subplots(figsize=(8, 6))
    vm_avg = np.percentile(np.abs(virt_lfp_final), 99)
    
    im2 = ax2.imshow(virt_lfp_final, aspect='auto', cmap='RdBu_r', origin='lower', 
                     vmin=-vm_avg, vmax=vm_avg,
                     extent=[time_ax[0], time_ax[-1], depth_axis[0], depth_axis[-1]])
    ax2.axhline(ssc_sink_depth, color='cyan', lw=2, linestyle='--', label=f'Sink @ {ssc_sink_depth:.0f}µm')
    ax2.axvline(0, color='Black', lw=2, linestyle='--', label=f'Stimulus onset')
    ax2.set_title("Averaged Evoked LFP")
    ax2.set_xlabel("Time from Stimulus (s)")
    ax2.set_ylabel("Depth (µm)")
    ax2.legend(loc='upper right')
    fig3.colorbar(im2, ax=ax2, label="Voltage (mV)")
    save_pub_fig(fig3, "3_Avg_LFP", mouse, save_dir)
    
    # ==========================================
    # Plot 4: Current Source Density (CSD)
    # ==========================================
    fig4, ax3 = plt.subplots(figsize=(8, 6))
    vm_csd = np.percentile(np.abs(csd), 99)
    
    im3 = ax3.imshow(csd, aspect='auto', cmap='viridis', origin='lower', 
                     vmin=-vm_csd, vmax=vm_csd,
                     extent=[time_ax[0], time_ax[-1], depth_csd[0], depth_csd[-1]])
    ax3.axhline(ssc_sink_depth, color='cyan', lw=2, linestyle='--', label='Primary Sink')
    ax3.axvline(0, color='black', lw=2, linestyle='--', label=f'Stimulus onset')
    ax3.set_title("Current Source Density (CSD)")
    ax3.set_xlabel("Time from Stimulus (s)")
    ax3.set_ylabel("Depth (µm)")
    ax3.legend(loc='upper right')
    fig4.colorbar(im3, ax=ax3, label="CSD (mV/mm²)")
    save_pub_fig(fig4, "4_CSD", mouse, save_dir)
    
    # ==========================================
    # Plot 5: Anatomy vs Physiology (Unit Spikes)
    # ==========================================
    fig5, ax4 = plt.subplots(figsize=(10, 6))
    vis_units = df_res[(df_res['sig']) & (df_res['modulation']>0)]
    motor_units = df_res[~df_res['sig']]
    
    # Scatter plots for units
    ax4.scatter(motor_units['depth'], motor_units['modulation'], 
                c='gray', alpha=0.3, label='Non-Responsive', edgecolors='none')
    ax4.scatter(vis_units['depth'], vis_units['modulation'], 
                c='red', alpha=0.8, label='Visual Units', edgecolors='white', linewidths=0.5)
    
    # Density plotting
    if len(vis_units) > 1:
        y_grid = np.linspace(PARAMS['probe_tip_offset'], np.max(depth_map), 200)
        kde = gaussian_kde(vis_units['depth'], bw_method=0.1)
        dens = kde(y_grid)
        
        # Create a secondary axis for the density plot to keep scales distinct
        ax4_dens = ax4.twinx()
        ax4_dens.plot(y_grid, dens, 'k--', lw=2, label='Visual Unit Density')
        ax4_dens.set_ylabel("Density", color='k')
        ax4_dens.spines['top'].set_visible(False)
    
    # Landmarks
    ax4.axvline(ssc_sink_depth, color='cyan', lw=3, label='LFP Sink (Superficial)')
    
    ax4.set_title("KDE of visually responsive units")
    ax4.set_xlabel("Probe Depth (µm)")
    ax4.set_ylabel("Modulation Index")
    
    # Combining legends from both axes if twinx was used
    lines_1, labels_1 = ax4.get_legend_handles_labels()
    if len(vis_units) > 1:
        lines_2, labels_2 = ax4_dens.get_legend_handles_labels()
        ax4.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')
    else:
        ax4.legend(loc='upper left')
    
    save_pub_fig(fig5, "5_KDE_spikes", mouse, save_dir)

    # ==========================================
    # Plot 6: Spiking MUA (Spike Density Map)
    # ==========================================
    mua_map, t_ax, d_ax = calculate_spiking_mua_depth(st, sp, triggers)
    fig6, ax5 = plt.subplots(figsize=(8, 6))
    
    vm_spiking = np.percentile(mua_map, 99) # Cap the maximum color at the 99.5th percentile to prevent blowout from massive individual units
    
    im5 = ax5.imshow(mua_map, aspect='auto', cmap='magma', origin='lower', 
                     vmin=0, vmax=vm_spiking,
                     extent=[t_ax[0], t_ax[-1], d_ax[0], d_ax[-1]])
    
    # Overlay the previously identified CSD sink for anatomical correlation
    ax5.axhline(ssc_sink_depth, color='cyan', lw=2, linestyle='--', label=f'LFP Sink @ {ssc_sink_depth:.0f}µm')
    ax5.axvline(0, color='white', lw=2, linestyle='--', label=f'Stimulus Onset')
    
    ax5.set_title("Evoked Spiking Multi-Unit Activity (Spike Density)")
    ax5.set_xlabel("Time from Checkerboard Reversal (ms)")
    ax5.set_ylabel("Probe Depth (µm)")
    ax5.legend(loc='upper right')
    fig6.colorbar(im5, ax=ax5, label="Firing Rate (Spikes/s)")
    
    save_pub_fig(fig6, "6_Spiking_MUA_Map", mouse, save_dir)
    # ==========================================
    # Plot 7: psd
    # ==========================================
    
    freq_mask = (frequencies >= 1) & (frequencies <= 100)
    freqs_plot = frequencies[freq_mask]
    psd_plot = psd[:, freq_mask]

    
    fig7, ax6 = plt.subplots(figsize=(8, 6))
    mesh = ax6.pcolormesh(
        freqs_plot, 
        depth_axis, 
        psd_plot, 
        shading='auto', 
        cmap='viridis',
        norm=LogNorm() # Log-scale normalization for power
    )
    

    ax6.set_xlabel('Frequency (Hz)')
    ax6.set_ylabel('Depth ($\mu m$)')
    ax6.set_title('Local Field Potential Power Spectral Density')
    ax6.axhline(ssc_sink_depth, color='cyan', lw=2, linestyle='--', label=f'LFP Sink @ {ssc_sink_depth:.0f}µm')
    ax6.legend(loc='upper right')
    cbar = fig7.colorbar(mesh, ax=ax6, label='Power ($V^2/Hz$)')
    save_pub_fig(fig7, "7_PSD", mouse, save_dir)



    #Waveforms all channels
    num_channels, num_samples = mean_clean.shape
    time_vector = np.linspace(-0.1, 0.5, num_samples)
    depths = depth_map
    # 4. Set up the colormap based on depth
    # We normalize the depth values to a range of 0 to 1 for the colormap
    norm = Normalize(vmin=np.min(depths), vmax=np.max(depth_axis))
    cmap = plt.get_cmap('coolwarm') # 'viridis', 'plasma', or 'coolwarm' work well
    
    # 5. Create the plot
    fig8, ax7 = plt.subplots(figsize=(10, 8))

    # 6. Loop through each channel and plot
    for i in range(num_channels):
        # Get the depth for this specific channel
        channel_depth = depths[i]
        
        # Get the specific color for this depth
        color = cmap(norm(channel_depth))
        
        # Plot the waveform
        ax7.plot(time_vector, mean_clean[i, :], color=color, alpha=0.7, linewidth=0.8)
    
    # 7. Add a vertical line at t=0 (stimulus onset)
    ax7.axvline(x=0, color='red', linestyle='--', label='Stimulus Onset')
    
    # 8. Formatting the axes and labels
    ax7.set_title('Mean LFP Waveforms Across Channels', fontsize=14)
    ax7.set_xlabel('Time relative to stimulus (s)', fontsize=12)
    ax7.set_ylabel('Voltage (\u00B5V)', fontsize=12)
    ax7.set_xlim([-0.1, 0.5])
    
    # 9. Add a colorbar to act as a depth legend
    # We create a ScalarMappable to link the colormap and the normalization
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([]) # Required for older versions of matplotlib
    cbar = fig8.colorbar(sm, ax=ax7)
    cbar.set_label('Probe Depth (\u00B5m)', rotation=270, labelpad=15)
    save_pub_fig(fig8, "8_Waveforms", mouse, save_dir)


    #Virtual LFP waveforms by depth
    num_channels, num_samples = virt_lfp_final.shape
    time_vector = np.linspace(-0.1, 0.5, num_samples)
    depths = depth_axis
    # 4. Set up the colormap based on depth
    # We normalize the depth values to a range of 0 to 1 for the colormap
    norm = Normalize(vmin=np.min(depths), vmax=np.max(depth_axis))
    cmap = plt.get_cmap('coolwarm') # 'viridis', 'plasma', or 'coolwarm' work well
    
    # 5. Create the plot
    fig9, ax8 = plt.subplots(figsize=(10, 8))
    
    # 6. Loop through each channel and plot
    for i in range(num_channels):
        # Get the depth for this specific channel
        channel_depth = depths[i]
        
        # Get the specific color for this depth
        color = cmap(norm(channel_depth))
        
        # Plot the waveform
        ax8.plot(time_vector, virt_lfp_final[i, :], color=color, alpha=0.7, linewidth=0.8)
    
    # 7. Add a vertical line at t=0 (stimulus onset)
    ax8.axvline(x=0, color='red', linestyle='--', label='Stimulus Onset')
    
    # 8. Formatting the axes and labels
    ax8.set_title('Mean LFP Waveforms Across Channels', fontsize=14)
    ax8.set_xlabel('Time relative to stimulus (s)', fontsize=12)
    ax8.set_ylabel('Voltage (\u00B5V)', fontsize=12)
    ax8.set_xlim([-0.1, 0.5])
    
    # 9. Add a colorbar to act as a depth legend
    # We create a ScalarMappable to link the colormap and the normalization
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([]) # Required for older versions of matplotlib
    cbar = fig9.colorbar(sm, ax=ax8)
    cbar.set_label('Probe Depth (\u00B5m)', rotation=270, labelpad=15)
    save_pub_fig(fig9, "9_Waveforms_virtual", mouse, save_dir)
    
    return {'lfp_virt': virt_lfp_final, 
            'csd': csd, 
            'units': df_res, 
            'anatomy': ssc_sink_depth, 
            "depth_axis": depth_axis, 
            "lfp": mean_clean, 
            "depths": depth_map, 
            "SEM": mean_sem}

In [3]:
def find_sc_channel(data, params, depth, depth_axis):
    """
    function to find channel id by depth, uses depth from virtual probe
    takes calculated depth from main pipeline find channels within a 100 micron bin around that depth
    isolates data with said channels and time mask of window
    finds most negative
    returns channel id of most negative. Essentially unpacks the virtual probe so instead of finding best depth of sSc (as per the sSc_sink_depth) variable
    it finds best channel around this identified depth and reports the depth of the probe (i.e unpacking the laminarity)
    """
    #data not at "0" as lfp averaging takes data before trigger, need to offset window
    offset = params["window_lfp"][0]
    window = params["window_visual"]
    start = window[0] + abs(offset)
    end = window[1] + abs(offset)
    fs = params["fs_lfp"]
    
    start = int(start * fs)
    end = int(end * fs)
    
    d_max = depth + 100
    d_min = depth - 100
    depth_mask = (depth_axis >= d_min) & (depth_axis <= d_max)
    subset = data[depth_mask, start:end]
    
    index_min = np.nanargmin(subset)
    row, col = np.unravel_index(index_min, subset.shape)

    valid_ids = np.where(depth_mask)[0]

    best_channel = valid_ids[row]
    best_depth = depth_axis[best_channel]
    
    best_time = (start + col) / fs + offset
    volt = subset[row, col]
    print(f"best channel ID {best_channel} @ {best_depth} @ {best_time}ms")
    print(f"nVEP is {volt}")
    return best_channel, best_depth, best_time, volt
    

In [4]:
def map_probe_sc_centered(channel_positions, best_channel_idx, 
                          entry_coords, terminus_coords, 
                          atlas_name='allen_mouse_25um', resolution_step=10):
    
    # --- 1. SETUP ---
    channel_x = channel_positions[:, 0] - 758
    channel_depths = channel_positions[:, 1]
    
    atlas = BrainGlobeAtlas(atlas_name)
    
    # --- 2. TRAJECTORY SAMPLING ---
    p0 = np.array(entry_coords, dtype=float)
    p1 = np.array(terminus_coords, dtype=float)
    traj_vec = p1 - p0
    traj_len = np.linalg.norm(traj_vec)
    traj_unit = traj_vec / traj_len
    
    steps = np.arange(0, traj_len, resolution_step)
    sample_points = p0 + (steps[:, np.newaxis] * traj_unit)
    
    # --- 3. FIND SUPERFICIAL SC INTERSECTION ---
    scs_indices = []
    
    # Define the target ID for the superficial/sensory SC
    # In Allen CCF, 'SCs' (Superior colliculus, sensory related) is usually ID 302.
    # You can also pass a list of IDs if targeting specific layers like SCop, SCsg, SCzo
    superficial_target_ids = [302] 
    
    # Pre-fetch hierarchy for speed
    for i, pt in enumerate(sample_points):
        try:
            sid = atlas.structure_from_coords(pt, microns=True)
            if sid > 0:
                # Check ancestry: Is this point inside ANY of the superficial target structures?
                is_superficial = any(
                    sid == target_id or atlas.hierarchy.is_ancestor(target_id, sid) 
                    for target_id in superficial_target_ids
                )
                
                if is_superficial:
                    scs_indices.append(i)
        except IndexError:
            continue
    
    if not scs_indices:
        raise ValueError("Trajectory does not pass through the Superficial Superior Colliculus.")
    
    # Calculate Superficial SC Geometry
    scs_entry = sample_points[scs_indices[0]]
    scs_exit = sample_points[scs_indices[-1]]
    
    # Geometrical center of the superficial segment
    scs_center_coords = (scs_entry + scs_exit) / 2.0
    scs_span = np.linalg.norm(scs_exit - scs_entry)
    
    print(f"Superficial SC Intersection: Span {scs_span:.0f} um")
    print(f"Superficial SC Center Coords: {scs_center_coords}")
    
    # --- 4. MAP PROBE GEOMETRY ---

    facing_vec = np.array([-1, 0, 0]) 
    lateral_vec = np.cross(traj_unit, facing_vec)
    lateral_vec /= np.linalg.norm(lateral_vec)
    #lateral_vec = -lateral_vec
    
    anchor_depth = channel_depths[best_channel_idx]
    dist_from_anchor = channel_depths - anchor_depth
    
    channel_coords = (scs_center_coords 
                      - (dist_from_anchor[:, np.newaxis] * traj_unit)
                      + (channel_x[:, np.newaxis] * lateral_vec))
    """
    
    target_ml_axis = np.array([0.0, 0.0, -1.0]) 
    
    # 2. Gram-Schmidt Orthogonalization to build a mathematically perfect lateral vector
    dot_product = np.dot(target_ml_axis, traj_unit)
    lateral_vec = target_ml_axis - (dot_product * traj_unit)
    lateral_vec /= np.linalg.norm(lateral_vec)
    
    # 3. Calculate channel distances from the top of the track (or your chosen anchor)
    dist_from_top = channel_depths - channel_depths[0]
    
    # 4. Generate the final 3D coordinates for all channels
    channel_coords = (entry_coords
                      + (dist_from_top[:, np.newaxis] * traj_unit) 
                      + (channel_x[:, np.newaxis] * lateral_vec))

    """
    
    # --- 5. ATLAS LOOKUP FOR CHANNELS ---
    regions = []
    acronyms = []
    
    for coord in channel_coords:
        try:
            sid = atlas.structure_from_coords(coord, microns=True)
            if sid > 0:
                acr = atlas.structures[sid]['acronym']
            else:
                sid = 0
                acr = "Void"
        except IndexError:
            sid = 0
            acr = "Out"
        regions.append(sid)
        acronyms.append(acr)
        
    # --- 6. OUTPUT ---
    df = pd.DataFrame({
        'channel_id': np.arange(len(channel_positions)),
        'rel_x': channel_x,
        'rel_y': channel_depths,
        'ccf_ap': channel_coords[:, 0],
        'ccf_dv': channel_coords[:, 1],
        'ccf_ml': channel_coords[:, 2],
        'region_id': regions,
        'acronym': acronyms
    })
    
    sc_info = {
        'entry': scs_entry,
        'exit': scs_exit,
        'center': scs_center_coords,
        'span_um': scs_span
    }
    
    return df, sc_info

In [5]:
def plot_channel_sem(target_channel, meanlfp, semlfp):
    
    num_channels, num_samples = meanlfp.shape
    
    # 2. Generate the time vector
    # 600 samples at 1000 Hz spans exactly 0.6 seconds.
    time_vector = np.linspace(-0.1, 0.5, num_samples)
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot the central mean waveform
    ax.plot(time_vector, meanlfp[target_channel, :], color='black', alpha=1, linewidth=1.5, label='Mean LFP')
    
    # Plot the SEM as a shaded region using the loaded semlfp array
    ax.fill_between(time_vector, 
                     meanlfp[target_channel, :] - semlfp[target_channel, :], 
                     meanlfp[target_channel, :] + semlfp[target_channel, :], 
                     color='gray', alpha=0.3, label='$\pm$ 1 SEM')
    
    # Add stimulus onset marker
    ax.axvline(x=0, color='red', linestyle='--', label='Checkerboard Onset')
    
    # Formatting the axes and labels
    ax.set_title(f'Mean LFP Waveform (Channel {target_channel})', fontsize=14)
    ax.set_xlabel('Time relative to stimulus (s)', fontsize=12)
    ax.set_ylabel('Voltage (\u00B5V)', fontsize=12)
    ax.set_xlim([-0.1, 0.5])
    
    ax.legend(loc='upper right')
    return fig, ax


In [11]:

if __name__ == "__main__":
    Mouse = []
    best_channels = []
    sc_depths = []
    latencies = []
    nVEPS = []
    mice = [10, 11]
    for i in mice:
        print("Analysing Mouse " + str(i))
        FILE_PATH = cfad.data['Neural']['Renewal']['Concatenated Data']['Mouse ' + str(i)] + r"/mouse" + str(i) + r"_renewal_concatenated_neural_data.dat"
        KS_DIR = cfad.data['Neural']['Renewal']['Concatenated Data']['kilosort4']['Mouse ' + str(i)]
        checker_triggers = np.array(scipy.io.loadmat(cfad.data['Neural']['Renewal']['Triggers']['Checkerboard']['Mouse ' + str(i)])["evt"]).flatten()
        OUTPUT_FOLDER = r'Z:\Saij\Data\Checkerboard\ScMouse' + str(i) 
        TRIGGERS = checker_triggers[1:]
        #probe_path = r"Z:\Saij\Herbs probes\Mouse " + str(i) + r"\probe_ccf_vox.npy" 
        #probe = np.load(probe_path)
        #probe *= 25
        
        results = run_sc_pipeline(FILE_PATH, KS_DIR, TRIGGERS, i, save_dir=OUTPUT_FOLDER, force_compute = False)
        depths = get_depth_map_full(KS_DIR)
        scID, scDepth, latency, volt = find_sc_channel(results["lfp"], PARAMS, results["anatomy"], results["depths"])
        Mouse.append(i)
        best_channels.append(scID)
        sc_depths.append(scDepth)
        latencies.append(latency)
        nVEPS.append(volt)

        fig, ax = plot_channel_sem(scID, results['lfp'], results['SEM'])
        save_pub_fig(fig, "10_Best_Channel", i, OUTPUT_FOLDER)
        #df, info = map_probe_sc_centered(depths, scID, probe[0], probe[1])
    
        #final_output_dir = r"Z:\Saij\ccf_channels"
        #df.to_csv(final_output_dir + r"\Mouse " + str(i) + r".csv", index = False)
    Group = [None, "Vehicle", "Vehicle", "Psilocybin", "Vehicle", "Psilocybin", "Psilocybin", "Psilocybin", "Vehicle",]
    """
    
    vep_df = pd.DataFrame({
        "Mouse": Mouse,
        "Sc Channel": best_channels,
        "Depth (µm)": sc_depths,
        "Latency (s)": latencies,
        "nVEP (mV)": nVEPS,
        "Treatment Group": Group[2:]
    })
    vep_df.to_csv(r"Z:\Saij\Data\Checkerboard\nVEP.csv", index = False)
    """


Analysing Mouse 10
--- PROCESSING: mouse10_renewal_concatenated_neural_data.dat ---
Step 1: Loading Probe Geometry...
Step 2: Calibrating (Detecting Bad Channels)...
  > Flagged 48 Bad Channels
Step 3: Extracting Diagnostic Sample...
Mean already computed, loading in as memmap
Step 5: Creating Virtual Probe (Average)...
Saved LFP binary to Z:\Saij\Data\Checkerboard\ScMouse10\data\mean_virtualLFP\mean_virtualLFP.dat
Saved LFP metadata to Z:\Saij\Data\Checkerboard\ScMouse10\data\mean_virtualLFP\mean_virtualLFP.json
Step 6: Calculating CSD...
  > sSC Center: 2940.0 um
Step 7: Integrating Spikes...
Generating Standalone Plots...
Saved: Mouse10_1_Raw_Continuous
Saved: Mouse10_2_Virtual_Continuous
Saved: Mouse10_3_Avg_LFP
Saved: Mouse10_4_CSD
Saved: Mouse10_5_KDE_spikes
Calculating depth-resolved Spike Density MUA...
MUA Spiking map generated.
Saved: Mouse10_6_Spiking_MUA_Map
Saved: Mouse10_7_PSD
Saved: Mouse10_8_Waveforms
Saved: Mouse10_9_Waveforms_virtual
best channel ID 251 @ 2955.0 @ 0.1